# Verifikasi kandidat SENA v0.6

Ekspor hanya setelah training selesai. Pemeriksaan kontrak dan inferensi CPU memakai hash bobot/config yang eksplisit. Parameter tidak diubah selama evaluasi ini. Acuan manusia lama 50 dan 24 usulan yang sudah diketahui merupakan regresi pengembangan. Lima puluh narasi baru telah dibekukan sebelum training dan hanya diuji sekali setelah model dipilih. Acuan baru masih berlabel asisten, sehingga metriknya tidak menjadi persetujuan mutu manusia otomatis. Tidak ada training atau fitting dalam notebook ini.

In [ ]:
from pathlib import Path
import os,sys,json,subprocess
ROOT=Path(os.environ['SLR09_PACKAGE_ROOT'])
WORK=Path(os.environ['SLR09_WORK_DIR'])
RUN=WORK/'postfit_scope_v06'
OUTPUT=WORK/'quality_diagnostics_v06'
def execute(command):
 p=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
 for line in p.stdout: print(line,end='',flush=True)
 p.wait()
 if p.returncode: raise RuntimeError('Step failed: '+str(p.returncode))
assert json.loads((RUN/'run_state.json').read_text())['status']=='SUCCEEDED'

In [ ]:
execute([sys.executable,str(ROOT/'code/quality_fix/prepare_scope_candidate_v06.py'),
 '--run',str(RUN),'--output',str(WORK/'exports')])

In [ ]:
execute([sys.executable,str(ROOT/'code/run_active_tests.py'),
 '--evidence-dir',str(WORK/'contract_tests_v06')])

In [ ]:
execute([sys.executable,str(ROOT/'code/quality_fix/run_scope_diagnostics_v06.py'),
 '--previous',str(ROOT/'evidence/quality_fix_v05/previous_50'),
 '--seen',str(ROOT/'data/quality_fix_fresh_challenge_v0.5.json'),
 '--fresh',str(ROOT/'data/frozen_new_diagnostic_v0.6.json'),
 '--sena-dir',str(RUN/'best_model'),
 '--nara-archive',str(WORK/'assets/CT/SILAR_NARA_reviewed_pilot_20261018T001122_607337Z.zip'),
 '--training-run',str(RUN),'--output',str(OUTPUT)])

In [ ]:
state=json.loads((OUTPUT/'run_state.json').read_text())
assert state['status']=='SUCCEEDED' and state['model_inference_executed']
assert not state['parameter_fitting_executed'] and not state['quality_acceptance_approved']
print(json.dumps(state['diagnostic_summary'],indent=2))
print('Human acceptance remains pending; no production deployment.')